# Modelo KMeans para asignar rangos a los indicadores del modelo CAMEL

In [6]:
import pandas as pd
from sklearn.cluster import KMeans

In [5]:
data = pd.read_excel('../tablas/registros_categorizados_con_IRL.xlsx')
data

,ID_registro,ID_indicador,ID_cooperativa,ano,mes,valor,categoria
0,25345,Quebranto Patrimonial,CAJA COOPERATIVA CREDICOOP,2020,1,0.000000e+00,Medianas
1,25346,Quebranto Patrimonial,CAJA COOPERATIVA PETROLERA,2020,1,0.000000e+00,Megas
2,25347,Quebranto Patrimonial,CASA NACIONAL DEL PROFESOR,2020,1,0.000000e+00,Megas
3,25348,Quebranto Patrimonial,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,2020,1,0.000000e+00,Grandes
4,25349,Quebranto Patrimonial,COOPERATIVA AHORRO Y CREDITO GOMEZ PLATA LTDA.,2020,1,0.000000e+00,Medianas
...,...,...,...,...,...,...,...
105855,99,IRL,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO ...,2025,8,7.624826e+08,Micro 2
105856,99,IRL,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO ...,2025,9,9.911326e+08,Micro 2
105857,99,IRL,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO ...,2025,10,9.842681e+08,Micro 2
105858,99,IRL,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO ...,2025,11,5.706928e+08,Micro 2


In [ ]:
import matplotlib.pyplot as plt

indicadores = data['ID_indicador'].unique()
categorias = data['categoria'].unique()

for indicador in indicadores:

    for categoria in categorias:

        subset = data[
            (data['ID_indicador'] == indicador) &
            (data['categoria'] == categoria)
        ]

        if len(subset) < 10:
            continue

        X = subset[['valor']]

        wcss = []

        K = range(1, 11)

        for k in K:

            kmeans = KMeans(
                n_clusters=k,
                random_state=42
            )

            kmeans.fit(X)

            wcss.append(kmeans.inertia_)

        plt.figure(figsize=(6,4))

        plt.plot(K, wcss, marker='o')

        plt.title(f'{indicador} - {categoria}')

        plt.xlabel('k')
        plt.ylabel('WCSS')

        plt.show()

## Pasos:

1. Separar los indicadores del modelo y las categorías
2. Recorrer cada uno de ellos y filtrar el subconjunto correspondiente. Ej: (Quebranto Patrimonial, Medianas)
3. Validar cantidad mínima de datos dentro del conjunto
4. Extraer el valor para el subset
5. Crear modelo KMeans 
6. Ajustar el modelo y asignar clusters (con los centroides internamente)
7. Ordenar los clusters desde el menor valor hasta el mayor (se impone esto para organizar y determinar qué tan bueno es en este contexto financiero). Con ello se estable el score para cada uno.
8. Crear diccionario de equivalencia
9. Convertir el cluster técnico en una calificación financiera interpretable. Ej: El orden de los clusters quedó 1, 2 y 0, con scores de 1, 2, 3 respectivamente; indicando la calidad de ese valor de calificación.

In [14]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

resultados = []

indicadores = data['ID_indicador'].unique()
categorias = data['categoria'].unique()

for indicador in indicadores:

    for categoria in categorias:

        subset = data[
            (data['ID_indicador'] == indicador) &
            (data['categoria'] == categoria)
        ].copy()

        # evitar grupos pequeños
        if len(subset) < 10:
            continue

        X = subset[['valor']]

        # BUSCAR MEJOR K
        mejor_k = 2
        mejor_score = -1

        # no puede haber más clusters
        # que cantidad de registros
        max_k = min(10, len(subset) - 1)

        for k in range(2, max_k + 1):
            modelo_temp = KMeans(
                n_clusters=k,
                random_state=42
            )

            labels = modelo_temp.fit_predict(X)

            score = silhouette_score(X, labels)

            if score > mejor_score:
                mejor_score = score
                mejor_k = k

        kmeans = KMeans(
            n_clusters=mejor_k,
            random_state=42
        )

        subset['cluster'] = kmeans.fit_predict(X)

        centroides = pd.DataFrame({
            'cluster': range(mejor_k),
            'centroide': kmeans.cluster_centers_.flatten()
        })

        centroides = centroides.sort_values(
            'centroide'
        ).reset_index(drop=True)

        centroides['score'] = range(1, mejor_k + 1)

        mapa = dict(
            zip(
                centroides['cluster'],
                centroides['score']
            )
        )

        subset['score'] = subset['cluster'].map(mapa)

        # guardar mejor k encontrado
        subset['k_optimo'] = mejor_k

        resultados.append(subset)

# unir resultados finales
df_final = pd.concat(resultados, ignore_index=True)

In [15]:
rangos = (
    df_final
    .groupby(
        ['ID_indicador', 'categoria', 'score']
    )['valor']
    .agg(['min', 'max'])
    .reset_index()
)

In [16]:
df_final

,ID_registro,ID_indicador,ID_cooperativa,ano,mes,valor,categoria,cluster,score,k_optimo
0,25345,Quebranto Patrimonial,CAJA COOPERATIVA CREDICOOP,2020,1,0.000000e+00,Medianas,1,1,3
1,25349,Quebranto Patrimonial,COOPERATIVA AHORRO Y CREDITO GOMEZ PLATA LTDA.,2020,1,0.000000e+00,Medianas,1,1,3
2,25350,Quebranto Patrimonial,COOPERATIVA BELEN AHORRO Y CREDITO,2020,1,0.000000e+00,Medianas,1,1,3
3,25352,Quebranto Patrimonial,COOPERATIVA DE AHORRO Y CREDITO BERLIN,2020,1,0.000000e+00,Medianas,1,1,3
4,25353,Quebranto Patrimonial,COOPERATIVA DE AHORRO Y CREDITO COLANTA,2020,1,0.000000e+00,Medianas,1,1,3
...,...,...,...,...,...,...,...,...,...,...
105855,99,IRL,COOPERATIVA FINANCIERA CAFETERA,2025,8,1.001089e+10,Top 4,8,6,9
105856,99,IRL,COOPERATIVA FINANCIERA CAFETERA,2025,9,1.141911e+10,Top 4,4,7,9
105857,99,IRL,COOPERATIVA FINANCIERA CAFETERA,2025,10,7.620293e+09,Top 4,6,3,9
105858,99,IRL,COOPERATIVA FINANCIERA CAFETERA,2025,11,9.461097e+09,Top 4,0,5,9


In [18]:
df_final.to_excel('../tablas/registros_categorizados_y_KMeans.xlsx', index=False)

In [17]:
rangos

,ID_indicador,categoria,score,min,max
0,Activo Productivo,Grandes,1,0.000000,0.000000
1,Activo Productivo,Grandes,2,0.679397,0.969199
2,Activo Productivo,Medianas,1,0.000000,0.000000
3,Activo Productivo,Medianas,2,0.563125,1.036441
4,Activo Productivo,Megas,1,0.000000,0.000000
...,...,...,...,...,...
365,Relación entre el Capital Institucional y el A...,Pequeñas,2,0.053293,0.099028
366,Relación entre el Capital Institucional y el A...,Pequeñas,3,0.100542,0.160992
367,Relación entre el Capital Institucional y el A...,Pequeñas,4,0.164112,0.217825
368,Relación entre el Capital Institucional y el A...,Top 4,1,0.000000,0.000000


In [19]:
rangos.to_excel('../tablas/rangos_KMeans.xlsx', index=False)